Alpha Leaf reduction setup

In [2]:
import pandas as pd
from hypernetx import Hypergraph

def build_hypergraph(edge_to_vertices):
    """
    Build X=(V,𝓔) from a hyperedge family 𝓔(X): edge_to_vertices[e] = set of vertices in e.
    Uses incidence table (v ∈ e) for stability.
    """
    rows = []
    for e, verts in edge_to_vertices.items():
        for v in verts:
            rows.append({"edges": e, "nodes": v, "weight": 1, "misc_properties": None})
    return Hypergraph(pd.DataFrame(rows))

def edge_sets(H):
    """Return 𝓔(H) as dict: e -> set(vertices in e)."""
    return {e: set(verts) for e, verts in H.incidence_dict.items()}

def print_hyperedges(H, title="Hyperedges"):
    print(title)
    for e, verts in edge_sets(H).items():
        print(f"  {e}: {sorted(verts)}")

def minimize_by_inclusion(H):
    """
    Safe minimization:
    If H has 0 or 1 hyperedges, return H.
    Otherwise, keep only inclusion-maximal hyperedges (toplexes).
    """
    E_map = edge_sets(H)
    if len(E_map) <= 1:
        return H
    return H.restrict_to_edges(H.toplexes())

def remove_size1_edges(H):
    """
    Enforce |e|>=2 for our setting.
    Safe on empty hypergraphs.
    """
    E_map = edge_sets(H)
    if not E_map:
        return H
    keep = [e for e, verts in E_map.items() if len(verts) >= 2]
    if not keep:
        # no hyperedges survive
        return H.restrict_to_edges([])  # produces an empty hypergraph object
    return H.restrict_to_edges(keep)

def vertex_incidence_degree(H):
    """deg_H(x) = number of hyperedges containing x."""
    deg = {}
    for e, verts in H.incidence_dict.items():
        for x in verts:
            deg[x] = deg.get(x, 0) + 1
    return deg

def alpha_leaves_and_parents(H):
    """
    α-leaf: deg=1
    parent: deg>=2
    """
    deg = vertex_incidence_degree(H)
    alpha = {x for x, d in deg.items() if d == 1}
    parents = {x for x, d in deg.items() if d >= 2}
    return alpha, parents, deg

def incident_hyperedges(H, x):
    """Return { e ∈ 𝓔(H) : x ∈ e }."""
    return [e for e, verts in H.incidence_dict.items() if x in verts]

WE BUILD ADJACENCY GRAPH BETWEEN INTERSECTING HYPEREDGES IN DUAL

In [3]:
def hyperedge_adjacency(E_map):
    """
    Build adjacency among hyperedges by nonempty intersection.
    Nodes: hyperedges (keys of E_map)
    Edge between e,f if E_map[e] ∩ E_map[f] ≠ ∅
    """
    edges = list(E_map.keys()) # list of hyperedges
    adj = {e: set() for e in edges} # dictionary of hyperedge -> set of its adjacent hyperedges
    for i in range(len(edges)): # loop over each hyperedge
        for j in range(i+1, len(edges)): # for each hyperedge, loop over subsequent hyperedges to avoid double counting
            e, f = edges[i], edges[j] # get the two hyperedges
            if E_map[e] & E_map[f]: # if they intersect (nonempty intersection)
                adj[e].add(f) # add f to e's adjacency set
                adj[f].add(e) # add e to f's adjacency set (undirected)
    return adj

def is_connected_hyperedge_graph(E_map):
    """
    Connectivity of the hyperedge-intersection graph.
    This is your 'are E1 and E3 still connected after deleting E2?' test.
    """
    if not E_map: # check if E_map is empty i.e., no hyperedges
        return True  # empty is vacuously connected for our purposes
    adj = hyperedge_adjacency(E_map) # get the adjacency list of the hyperedge graph
    start = next(iter(adj)) # get an arbitrary starting hyperedge default - (first key in the adjacency dictionary)
    seen = {start} # create a set to track seen hyperedges, initialized with the starting hyperedge
    stack = [start] # create a stack for searching, initialized with the starting hyperedge
    while stack: # while there are hyperedges to explore
        cur = stack.pop() # get the next hyperedge to explore0
        for nb in adj[cur]: # loop over the neighbors of the current hyperedge
            if nb not in seen: # if the neighbor hyperedge has not been seen yet
                seen.add(nb) # mark it as seen
                stack.append(nb) # add it to the stack for further exploration
    return len(seen) == len(adj) # check if all hyperedges were seen (i.e., the graph is connected)

Now we find the unique hyperedge an alpha belongs to

In [4]:
def unique_hyperedge_of_alpha_leaf(H, leaf):
    """
    α-leaf has exactly one incident hyperedge in M(H*).
    Return that hyperedge id.
    """
    inc = incident_hyperedges(H, leaf) # get the list of hyperedges that contain the leaf vertex
    if len(inc) != 1:
        raise ValueError(f"{leaf} is not an α-leaf (incident edges = {inc})")
    return inc[0]

We begin reduction attempt

In [5]:
def reduce_step(H_star_hat, clique_contents):
    """
    Attempt ONE safe reduction step on M(H*).
    Returns: (H_new, clique_contents_new, did_reduce, debug_info)
    """
    # get the hyperedge family as a dict: e -> set of vertices in e
    E_map = edge_sets(H_star_hat) 
    # get α-leaves, parents, and degree dict for vertices in H_star_hat
    alpha, parents, deg = alpha_leaves_and_parents(H_star_hat)

    # If no α-leaves, we cannot reduce since that means the graph is not helly
    if not alpha:
        return H_star_hat, clique_contents, False, {"reason": "no α-leaves"}

    # Pick an α-leaf (for now: deterministic choice by name)
    leaf = sorted(alpha, key=str)[0]
    # Find the unique hyperedge Ei that contains this α-leaf
    Ei = unique_hyperedge_of_alpha_leaf(H_star_hat, leaf)

    # Identify children in Ei = α-leaves contained in Ei
    children_in_Ei = sorted([x for x in E_map[Ei] if x in alpha], key=str)

    # Identify parents in Ei = vertices with deg>=2 contained in Ei
    parents_in_Ei = sorted([x for x in E_map[Ei] if x in parents], key=str)

    debug = {
        "chosen_leaf": leaf,
        "Ei": Ei,
        "children_in_Ei": children_in_Ei,
        "parents_in_Ei": parents_in_Ei,
        "case": None,
        "accepted": None
    }

    # If we delete Ei, will the remaining hyperedges still be connected?
    # we want to check if the leaf is good or bad
    # we create a a neew hyperedge family dict without Ei to check connectivity
    E_after_delete = {e: set(vs) for e, vs in E_map.items() if e != Ei}

    # we create the adjacency graph of the remaining hyperedges 
    # and check if it is connected
    good = is_connected_hyperedge_graph(E_after_delete)

    # if the check failed, then we know that the leaf is bad
    # and we reject this reduction step
    if not good:
        debug["accepted"] = False
        debug["reason"] = "deleting Ei disconnects hyperedge adjacency graph"
        return H_star_hat, clique_contents, False, debug

    # If it’s good, we ok the deletion of Ei plus parent-reduction, if needed
    # we create new clique contents dict to update for the reduction step
    clique_contents_new = {k: set(v) for k, v in clique_contents.items()}  # deep-ish copy

    # CASE R2: single child in Ei -> reduce ALL parents in Ei by removing leaf's content
    if len(children_in_Ei) == 1:
        debug["case"] = "single-child (R2: parent-content reduction)"

        # get the vertices that make the leaf clique 
        leaf_content = clique_contents_new.get(leaf, set())
        for p in parents_in_Ei:
            # Reduce parent clique-content: C(p) <- C(p) \ C(leaf)
            clique_contents_new[p] = clique_contents_new.get(p, set()) - leaf_content

        # (Cluster-control checks will happen AFTER recomputing H*, in a later function)
        # For now we only implement the core update + delete Ei.

    # CASE R1: multiple children in Ei -> do NOT reduce parents; just delete Ei
    else:
        debug["case"] = "multi-child (R1: no parent reduction)"

    # Delete Ei from hyperedge family
    H_new = H_star_hat.restrict_to_edges([e for e in E_map.keys() if e != Ei])

    # If no hyperedges remain, we're done; don't call toplexes()
    if len(edge_sets(H_new)) == 0:
        debug["accepted"] = True
        debug["reason"] = "all hyperedges deleted; stopping"
        return H_new, clique_contents_new, True, debug

    # Re-minimize and remove singletons again (your pipeline)
    H_new = minimize_by_inclusion(H_new)
    H_new = remove_size1_edges(H_new)

    debug["accepted"] = True
    return H_new, clique_contents_new, True, debug

TEST

In [6]:
# Toy H* hyperedges (already minimized + |e|>=2)
E_star_hat = {
    "E1": {"A","B","C","F"},
    "E2": {"C","E"}
}

# Toy clique contents (each clique-vertex corresponds to a maximal clique of G)
clique_contents = {
    "A": {"v1","v2","v3"},
    "B": {"v2","v4"},
    "C": {"v2","v5","v6"},
    "D": {"v5"},          # leaf-content
    "E": {"v6","v7"},
    "F": {"v8","v2"},
}

H_star_hat = build_hypergraph(E_star_hat)
H_star_hat = minimize_by_inclusion(H_star_hat)
H_star_hat = remove_size1_edges(H_star_hat)

print_hyperedges(H_star_hat, "Initial Ĥ*:")

for step in range(1, 6):
    H_star_hat, clique_contents, did, info = reduce_step(H_star_hat, clique_contents)
    print("\nStep", step, "| did_reduce =", did, "| info =", info)
    print_hyperedges(H_star_hat, "Ĥ* after step:")
    if not did:
        break

Initial Ĥ*:
  E1: ['A', 'B', 'C', 'F']
  E2: ['C', 'E']

Step 1 | did_reduce = True | info = {'chosen_leaf': 'A', 'Ei': 'E1', 'children_in_Ei': ['A', 'B', 'F'], 'parents_in_Ei': ['C'], 'case': 'multi-child (R1: no parent reduction)', 'accepted': True}
Ĥ* after step:
  E2: ['C', 'E']

Step 2 | did_reduce = True | info = {'chosen_leaf': 'C', 'Ei': 'E2', 'children_in_Ei': ['C', 'E'], 'parents_in_Ei': [], 'case': 'multi-child (R1: no parent reduction)', 'accepted': True, 'reason': 'all hyperedges deleted; stopping'}
Ĥ* after step:

Step 3 | did_reduce = False | info = {'reason': 'no α-leaves'}
Ĥ* after step:


Lets try adding Priority to reductions 

In [7]:
def classify_hyperedges_by_children(H_star_hat):
    """
    Return:
      - alpha: set of α-leaf vertices
      - parents: set of parent vertices
      - children_in_edge: dict e -> list of α-leaves in e
      - parents_in_edge:  dict e -> list of parents in e
    """
    E_map = edge_sets(H_star_hat)
    alpha, parents, deg = alpha_leaves_and_parents(H_star_hat)

    children_in_edge = {}
    parents_in_edge = {}
    for e, verts in E_map.items():
        children_in_edge[e] = sorted([x for x in verts if x in alpha], key=str)
        parents_in_edge[e]  = sorted([x for x in verts if x in parents], key=str)

    return alpha, parents, children_in_edge, parents_in_edge

We find the hyperedges with single childs as the preference

In [8]:
def pick_good_hyperedge_to_delete(H_star_hat, prefer_single_child=True):
    """
    Choose a hyperedge Ei to delete such that deleting Ei keeps the hyperedge-adjacency graph connected.
    Priority:
      - if prefer_single_child: try edges with exactly 1 child first
      - else: any good edge
    Returns: (Ei, case, children, parents) or (None, None, None, None)
      case = "single" or "multi"
    """
    E_map = edge_sets(H_star_hat)
    if len(E_map) <= 1: 
        # check if there are 0 or 1 hyperedges, 
        # in which case we can't delete any edge
        return None, None, None, None

    alpha, parents, children_in_edge, parents_in_edge = classify_hyperedges_by_children(H_star_hat)

    # check if a certain hyperedge can be safely deleted 
    def is_good_delete(Ei):
        E_after = {e: set(vs) for e, vs in E_map.items() if e != Ei}
        return is_connected_hyperedge_graph(E_after)

    # Candidate lists
    single_candidates = [e for e in E_map if len(children_in_edge[e]) == 1]
    multi_candidates  = [e for e in E_map if len(children_in_edge[e]) >= 2]

    # Deterministic ordering (so runs are reproducible)
    single_candidates = sorted(single_candidates, key=str)
    multi_candidates  = sorted(multi_candidates, key=str)

    if prefer_single_child: 
        # Try single-child edges first, then multi-child edges
        for e in single_candidates:
            # check if the hyperedge with single child is good to delete, if so return it
            if is_good_delete(e): 
                return e, "single", children_in_edge[e], parents_in_edge[e]
        for e in multi_candidates:
            # check if the hyperedge with multiple children is good to delete
            # not needed but we do it for completeness
            if is_good_delete(e):
                return e, "multi", children_in_edge[e], parents_in_edge[e]
    else:
        for e in single_candidates + multi_candidates:
            if is_good_delete(e):
                return e, "single" if len(children_in_edge[e]) == 1 else "multi", children_in_edge[e], parents_in_edge[e]

    return None, None, None, None

In [9]:
def execute_delete_step(H_star_hat, clique_contents, Ei, case, children, parents):
    """
    Delete Ei from H* and apply the correct update rule based on the case.
    Returns updated (H_new, clique_contents_new).
    """
    E_map = edge_sets(H_star_hat)

    clique_contents_new = {k: set(v) for k, v in clique_contents.items()}

    if case == "single":
        # exactly one child leaf L in Ei
        L = children[0]
        L_content = clique_contents_new.get(L, set())

        # Reduce ALL parents in Ei (per your latest rule)
        for p in parents:
            clique_contents_new[p] = clique_contents_new.get(p, set()) - L_content

    # case == "multi": no parent reductions

    # delete Ei
    H_new = H_star_hat.restrict_to_edges([e for e in E_map.keys() if e != Ei])

    # re-minimize + remove singletons (safe versions)
    H_new = minimize_by_inclusion(H_new)
    H_new = remove_size1_edges(H_new)

    return H_new, clique_contents_new

Lets try running full algorithm with priority reduction and guard plus movement number count

In [ ]:
def run_reduction_algorithm(H_star_hat, clique_contents):
    """
    PAPER OUTPUTS:
      k = guard count
      m = movement number
    Stop when only one hyperedge remains.
    Prioritize single-child deletions over multi-child deletions.
    """
    k = 0 # initial guard count
    m = 0 # initial movement count

    # Ensure we start in the proper "working form"
    H = minimize_by_inclusion(H_star_hat)
    H = remove_size1_edges(H)

    history = []

    while True:
        # get the hyperedge family as a dict: e -> set(vertices in e)
        E_map = edge_sets(H)

        # Stop condition: one hyperedge left
        if len(E_map) <= 1:
            break

        Ei, case, children, parents = pick_good_hyperedge_to_delete(H, prefer_single_child=True)

        if Ei is None:
            # No "good" deletion exists under our connectivity rule.
            # the graph is not helly, or we have a bug in our code, but we stop safely instead of crashing.
            history.append({"stopped": True, "reason": "no good cut available", "edges": list(E_map.keys())})
            break

        # Count guards and movement according to your rules
        k += 1 # default guard cost for any deletion
        if case == "multi":
            m += 1

        # Execute
        H, clique_contents = execute_delete_step(H, clique_contents, Ei, case, children, parents)

        history.append({
            "deleted_edge": Ei,
            "case": case,
            "children": children,
            "parents": parents,
            "k": k,
            "m": m,
            "remaining_edges": list(edge_sets(H).keys())
        })

    # Base case add-ons (if we ended with exactly one hyperedge)
    E_map = edge_sets(H)
    if len(E_map) == 1:
        last_edge = next(iter(E_map))
        # get the number of vertices in the last remaining hyperedge
        n = len(E_map[last_edge])

        # guards increment by 1 if n=1 else 2 (per your rule for the base case)
        k += 1 if n == 1 else 2 

        # movement increment by 1 if n<3 else 2 (per your rule for the base case)
        m += 1 if n < 3 else 2

        history.append({"base_case_edge": last_edge, "n": n, "k_final": k, "m_final": m})

    return k, m, H, clique_contents, history

Lets run a test

In [11]:
# --- Toy minimized dual H* (hyperedges -> clique-vertices) ---
E_star_hat = {
    "E1": {"A","B","C","F"},
    "E2": {"C","E"}
}

# --- Toy clique contents: each clique-vertex is a maximal clique of G ---
# (Used only for single-child parent reductions; doesn't affect connectivity itself.)
clique_contents = {
    "A": {"v1","v2","v3"},
    "B": {"v2","v4"},
    "C": {"v2","v5","v6"},
    "D": {"v5"},          # leaf content
    "E": {"v6","v7"},
    "F": {"v8","v2"},
}

# Build H from E_star_hat
H_star_hat = build_hypergraph(E_star_hat)
H_star_hat = minimize_by_inclusion(H_star_hat)
H_star_hat = remove_size1_edges(H_star_hat)

print_hyperedges(H_star_hat, "Initial H*:")

k, m, H_final, clique_contents_final, history = run_reduction_algorithm(H_star_hat, clique_contents)

print("\n=== RESULTS ===")
print("Guard count k =", k)
print("Movement number m =", m)
print_hyperedges(H_final, "Final hypergraph (base case stage):")

print("\n=== HISTORY (step-by-step) ===")
for i, h in enumerate(history, 1):
    print(f"{i}. {h}")

Initial H*:
  E1: ['A', 'B', 'C', 'F']
  E2: ['C', 'E']

=== RESULTS ===
Guard count k = 3
Movement number m = 2
Final hypergraph (base case stage):
  E1: ['A', 'B', 'C', 'F']

=== HISTORY (step-by-step) ===
1. {'deleted_edge': 'E2', 'case': 'single', 'children': ['E'], 'parents': ['C'], 'k': 1, 'm': 0, 'remaining_edges': ['E1']}
2. {'base_case_edge': 'E1', 'n': 4, 'k_final': 3, 'm_final': 2}
